##  ENTREGÁVEL 2: Integração com SQLite
Notebook para carregar os dados no banco

### Conexão com o banco de dados

In [ ]:
from pandas_utils import read_layer_csv
from db_utils import get_pg_connection, persist_dataframe, list_tables

conn = get_pg_connection()
print("Conexão com banco de dados PostgreSQL estabelecida")

### Carregamento da camada de dados Silver

In [ ]:
df_limpo = read_layer_csv("data/silver/dados_limpos.csv")
print(f"Dados Silver carregados: {len(df_limpo)} registros")

###  Salvamento da camada Silver no banco de dados

In [ ]:
persist_dataframe(df_limpo, "projeto_final", conn)

###  Carregamento dos dados agregados Gold

In [ ]:
metricas = read_layer_csv("data/gold/metricas_estado.csv")
ativos = read_layer_csv("data/gold/ativos_patrimonio.csv")

persist_dataframe(metricas, "metricas_estado", conn)
persist_dataframe(ativos, "ativos_patrimonio", conn)

### Criação do relacionamento entre as tabelas

In [ ]:
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS clientes (
    id_cliente INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    nome_cliente TEXT,
    email TEXT,
    cidade TEXT,
    estado TEXT
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS ativos (
    id_ativo INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    descricao TEXT,
    valor_atual DOUBLE PRECISION,
    categoria TEXT,
    id_cliente INTEGER,
    FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS metricas_estado (
    estado TEXT PRIMARY KEY,
    total_clientes INTEGER,
    media_patrimonio DOUBLE PRECISION,
    indice_atividade DOUBLE PRECISION
);
""")
conn.commit()
print("Tabelas GOLD estruturadas e relacionadas criadas com sucesso!")

### Verificação das tabelas criadas

In [ ]:
for table_name in list_tables(conn):
    print(f" - {table_name}")
conn.close()
print("Tabelas relacionadas criadas e salvas com sucesso!")